In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import joblib
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#폰트지정
plt.rcParams['font.family'] = 'Malgun Gothic'

#마이너스 부호 깨짐 지정
plt.rcParams['axes.unicode_minus'] = False

#숫자가 지수표현식으로 나올 때 지정
pd.options.display.float_format = '{:.2f}'.format

In [3]:
# 데이터 로드 및 기본 구조 확인

data = pd.read_csv('data/E-Commerce-Dataset.csv')

print("데이터 형태:", data.shape)
display(data.head())
data.info()

데이터 형태: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,160
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,121
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,130


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   object 
 13  NumberOfAddress   

In [4]:
# 히트맵 시각화1

df_clean = data.drop('CustomerID', axis=1).copy()

cat_cols = df_clean.select_dtypes(include=['object']).columns
le = LabelEncoder()

for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

plt.figure(figsize=(15, 10))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('이커머스 고객 이탈 데이터 상관관계 히트맵')
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\4147068657.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# 히트맵 시각화2

plt.figure(figsize=(10, 8))

numeric_cols = [
    'Churn', 'Tenure', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
    'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder', 'CashbackAmount'
]

corr = data[numeric_cols].corr()

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')

plt.title('상관관계 히트맵', fontsize=16, pad=15)
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\2484432823.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# 히트맵 시각화 3

plt.figure(figsize=(15, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm',
            vmin=-1, vmax=1, center=0, linewidths=0.5)
plt.title('이커머스 고객 이탈 데이터 상관관계 히트맵')
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\387999107.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# 데이터 전처리 및 분할

X = data.drop('Churn', axis=1)
y = data['Churn']

numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

categorical_cols = X.select_dtypes(exclude=[np.number]).columns

for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# 문자를 숫자로 변환 (다중공선성 방지를 위해 drop_first=True 적용)
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("전처리 완료된 데이터 컬럼 수:", X_train.shape[1], "개")
print("학습 데이터 크기:", X_train.shape)
print("테스트 데이터 크기:", X_test.shape)

전처리 완료된 데이터 컬럼 수: 30 개
학습 데이터 크기: (4504, 30)
테스트 데이터 크기: (1126, 30)


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report

# Logistic Regression

lr_model = LogisticRegression(max_iter=1000, random_state=42)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print("** Logistic Regression **")
print(f"정확도 (Accuracy): {lr_acc:.4f}")
print(f"정밀도 (Precision): {lr_precision:.4f}")
print(f"재현율 (Recall): {lr_recall:.4f}")
print(f"F1-Score: {lr_f1:.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_lr))

** Logistic Regression **
정확도 (Accuracy): 0.8863
정밀도 (Precision): 0.7422
재현율 (Recall): 0.5000
F1-Score: 0.5975

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.90      0.96      0.93       936
           1       0.74      0.50      0.60       190

    accuracy                           0.89      1126
   macro avg       0.82      0.73      0.77      1126
weighted avg       0.88      0.89      0.88      1126



c:\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
# features - Logistic Regression

features = [
    'Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
    'SatisfactionScore', 'NumberOfAddress', 'Complain',
    'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
    'DaySinceLastOrder', 'CashbackAmount'
]

"""
features = [
    'Tenure', 'WarehouseToHome', 'PreferredPaymentMode', 'PreferedOrderCat',
    'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain',
    'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount',
]
"""

X = data[features]
y = data['Churn']

X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("** Logistic Regression **")
print(f"정확도: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_lr):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_lr))

** Logistic Regression **
정확도: 0.8623
정밀도: 0.6496
재현율: 0.4000
F1-Score: 0.4951

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       936
           1       0.65      0.40      0.50       190

    accuracy                           0.86      1126
   macro avg       0.77      0.68      0.71      1126
weighted avg       0.85      0.86      0.85      1126



In [10]:
# features - Logistic Regression
"""
features = [
    'Tenure', 'WarehouseToHome', 'HourSpendOnApp', 'NumberOfDeviceRegistered',
    'SatisfactionScore', 'NumberOfAddress', 'Complain',
    'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
    'DaySinceLastOrder', 'CashbackAmount'
]
"""

features = [
    'Tenure', 'WarehouseToHome', 'PreferredPaymentMode', 'PreferedOrderCat',
    'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain',
    'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount',
]

X = data[features]
y = data['Churn']

numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

X.loc[:, numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

for col in categorical_cols:
    X.loc[:, col] = X[col].fillna(X[col].mode()[0])

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("** Logistic Regression **")
print(f"정확도: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_lr):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_lr))

** Logistic Regression **
정확도: 0.8837
정밀도: 0.7185
재현율: 0.5105
F1-Score: 0.5969

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.91      0.96      0.93       936
           1       0.72      0.51      0.60       190

    accuracy                           0.88      1126
   macro avg       0.81      0.73      0.76      1126
weighted avg       0.87      0.88      0.88      1126



c:\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [12]:
# Randomforest

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("** RandomForest **")
print(f"정확도: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_rf):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_rf))

** RandomForest **
정확도: 0.9680
정밀도: 0.9425
재현율: 0.8632
F1-Score: 0.9011

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       936
           1       0.94      0.86      0.90       190

    accuracy                           0.97      1126
   macro avg       0.96      0.93      0.94      1126
weighted avg       0.97      0.97      0.97      1126



In [13]:
# 모델 비교

results = {
    "Logistic Regression": {
        "Accuracy": accuracy_score(y_test, y_pred_lr),
        "Precision": precision_score(y_test, y_pred_lr),
        "Recall": recall_score(y_test, y_pred_lr),
        "F1-Score": f1_score(y_test, y_pred_lr)
    },
    "Random Forest": {
        "Accuracy": accuracy_score(y_test, y_pred_rf),
        "Precision": precision_score(y_test, y_pred_rf),
        "Recall": recall_score(y_test, y_pred_rf),
        "F1-Score": f1_score(y_test, y_pred_rf)
    }
}

# 딕셔너리를 DataFrame으로 변환 후 시각화
df_results = pd.DataFrame(results).T

ax = df_results.plot(kind='bar', figsize=(10, 6), width=0.7,
                     color=['skyblue', 'orange', 'lightgreen', 'pink'], edgecolor='black')

plt.title("로지스틱 회귀 vs 랜덤 포레스트 성능 비교", fontsize=15, pad=15)
plt.xlabel("모델", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.legend(title="평가 지표", loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for p in ax.patches:
    ax.annotate(f"{p.get_height():.4f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\1832977113.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# 전처리 데이터 확인

import xgboost as xgb
from sklearn.metrics import confusion_matrix

print("전처리 완료 데이터 확인")
display(X_train.head(3))

전처리 완료 데이터 확인


,Tenure,WarehouseToHome,SatisfactionScore,NumberOfAddress,Complain,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferredPaymentMode_COD,...,PreferredPaymentMode_Debit Card,PreferredPaymentMode_E wallet,PreferredPaymentMode_UPI,PreferedOrderCat_Grocery,PreferedOrderCat_Laptop & Accessory,PreferedOrderCat_Mobile,PreferedOrderCat_Mobile Phone,PreferedOrderCat_Others,MaritalStatus_Married,MaritalStatus_Single
1787,9.00,16.00,1,2,0,8.00,9.00,7.00,200,False,...,False,False,False,False,False,False,False,False,False,True
2147,6.00,13.00,4,1,0,0.00,1.00,2.00,143,False,...,True,False,False,False,True,False,False,False,True,False
1717,8.00,15.00,4,10,0,0.00,1.00,0.00,165,False,...,True,False,False,False,True,False,False,False,False,True


In [15]:
# XGBoost 모델 학습

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("** XGBoost **")
print(f"정확도: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_xgb):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_xgb):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_xgb):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_xgb))

c:\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [00:28:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


** XGBoost **
정확도: 0.9432
정밀도: 0.8841
재현율: 0.7632
F1-Score: 0.8192

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       936
           1       0.88      0.76      0.82       190

    accuracy                           0.94      1126
   macro avg       0.92      0.87      0.89      1126
weighted avg       0.94      0.94      0.94      1126



In [16]:
# XGBoost 모델 학습

scale_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"적용된 scale_pos_weight: {scale_weight:.2f}\n")
xgb_tuned_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_weight,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_tuned_model.fit(X_train, y_train)
y_pred_xgb_tuned = xgb_tuned_model.predict(X_test)

print("** XGBoost **")
print(f"정확도: {accuracy_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_xgb_tuned):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_xgb_tuned):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_xgb_tuned))

적용된 scale_pos_weight: 4.94



c:\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [00:28:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


** XGBoost **
정확도: 0.9494
정밀도: 0.7854
재현율: 0.9632
F1-Score: 0.8652

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       936
           1       0.79      0.96      0.87       190

    accuracy                           0.95      1126
   macro avg       0.89      0.95      0.92      1126
weighted avg       0.96      0.95      0.95      1126



In [17]:
#모델 비교

comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'] * 2,
    'Score': [
        0.9680, 0.9425, 0.8632, 0.9011,  # Random Forest 지표
        0.9494, 0.7854, 0.9632, 0.8652   # Tuned XGBoost 지표
    ],
    'Model': ['Random Forest'] * 4 + ['Tuned XGBoost'] * 4
}

df_compare = pd.DataFrame(comparison_data)

plt.figure(figsize=(10, 6))
sns.barplot(x='Metric', y='Score', hue='Model', data=df_compare, palette='Set2')

plt.title('Random Forest vs Tuned XGBoost 성능 지표 비교', fontsize=16, pad=15)
plt.ylim(0.7, 1.05)
plt.legend(loc='lower right', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)

ax = plt.gca()
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.4f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=10, fontweight='bold')

plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\1059386849.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# 최종 선택 : Randomforest

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("** RandomForest **")
print(f"정확도: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"정밀도: {precision_score(y_test, y_pred_rf):.4f}")
print(f"재현율: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print("\n[상세 분류 리포트]")
print(classification_report(y_test, y_pred_rf))

** RandomForest **
정확도: 0.9680
정밀도: 0.9425
재현율: 0.8632
F1-Score: 0.9011

[상세 분류 리포트]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       936
           1       0.94      0.86      0.90       190

    accuracy                           0.97      1126
   macro avg       0.96      0.93      0.94      1126
weighted avg       0.97      0.97      0.97      1126



In [19]:
# 혼동행렬

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['유지(0)', '이탈(1)'], yticklabels=['유지(0)', '이탈(1)'])

plt.title('Random Forest 혼동 행렬', fontsize=15, pad=15)
plt.xlabel('예측값 (Predicted Label)', fontsize=12)
plt.ylabel('실제값 (Actual Label)', fontsize=12)
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\3084893352.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Feature Importances

feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15), palette='viridis')

plt.title('Random Forest Feature Importances', fontsize=15, pad=15)
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\929867831.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15), palette='viridis')
C:\Users\82108\AppData\Local\Temp\ipykernel_42088\929867831.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# Prediction vs Actual

y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

plt.figure(figsize=(10, 6))

# 실제 유지 고객(0)의 예측 확률 분포 (파란색)
sns.histplot(y_pred_proba[y_test == 0], color='blue', label='실제 유지(0)',
             kde=True, stat='density', alpha=0.4, bins=30)

# 실제 이탈 고객(1)의 예측 확률 분포 (빨간색)
sns.histplot(y_pred_proba[y_test == 1], color='red', label='실제 이탈(1)',
             kde=True, stat='density', alpha=0.4, bins=30)

plt.title('실제 클래스별 이탈 예측 확률 분포', fontsize=15, pad=15)
plt.xlabel('이탈 예측 확률', fontsize=12)
plt.ylabel('밀도', fontsize=12)

plt.axvline(x=0.5, color='black', linestyle='--', label='임계값(0.5)')

plt.legend()
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\4195064346.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
print("변수별 결측치 개수")
print(data.isnull().sum())

print("\n전체 행 중복값 개수")
print(data.duplicated().sum())

변수별 결측치 개수
CustomerID                       0
Churn                            0
Tenure                         264
PreferredLoginDevice             0
CityTier                         0
WarehouseToHome                251
PreferredPaymentMode             0
Gender                           0
HourSpendOnApp                 255
NumberOfDeviceRegistered         0
PreferedOrderCat                 0
SatisfactionScore                0
MaritalStatus                    0
NumberOfAddress                  0
Complain                         0
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
CashbackAmount                   0
dtype: int64

전체 행 중복값 개수
0


In [23]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

key_vars = ['Tenure', 'DaySinceLastOrder', 'CashbackAmount', 'HourSpendOnApp']
titles = ['가입 기간 (Tenure)', '최근 구매일 (DaySinceLastOrder)', '누적 캐시백 (CashbackAmount)', '앱 체류 시간 (HourSpendOnApp)']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    var_data = data[var].dropna()

    mean_val = var_data.mean()
    median_val = var_data.median()
    std_val = var_data.std()

    sns.histplot(var_data, kde=True, ax=axes[i], color='cornflowerblue', bins=30)
    axes[i].set_title(f'{titles[i]} 분포', fontsize=14, pad=10)
    axes[i].set_ylabel('고객 수 (Count)')

    stats_text = f"평균: {mean_val:.2f}\n중앙값: {median_val:.2f}\n표준편차: {std_val:.2f}"
    props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9)

    axes[i].text(0.95, 0.95, stats_text, transform=axes[i].transAxes, fontsize=12,
                 verticalalignment='top', horizontalalignment='right', bbox=props)

plt.tight_layout()
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\381693648.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
# 학습 과정

from sklearn.model_selection import validation_curve

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

param_range = np.arange(10, 210, 20)

train_scores, test_scores = validation_curve(
    RandomForestClassifier(random_state=42),
    X_train, y_train,
    param_name="n_estimators",
    param_range=param_range,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(param_range, train_mean, marker='o', color='blue', label='훈련 데이터 F1-Score')
plt.plot(param_range, test_mean, marker='s', color='green', label='검증 데이터 F1-Score')

plt.title('랜덤 포레스트 모델 학습 곡선 (트리 개수에 따른 성능 변화)', fontsize=15, pad=15)
plt.xlabel('트리의 개수 (n_estimators)', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

C:\Users\82108\AppData\Local\Temp\ipykernel_42088\2321267652.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
import os

cat_cols = ['MaritalStatus', 'PreferedOrderCat']

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    le.fit(data[col].astype(str))
    encoders[col] = le

os.makedirs('data', exist_ok=True)
joblib.dump(rf_model, 'data/ecommerce_randomforest_model.pkl')
joblib.dump(encoders, 'data/ecommerce_labelencoders.pkl')

['data/ecommerce_labelencoders.pkl']